# SFT: All Legal Chess Moves in UCI Order

Fine-tune a text-generation model to return every legal UCI move for a FEN, sorted lexicographically. The base-model field accepts a Hugging Face model ID, a full local model directory, or a PEFT adapter directory containing `adapter_config.json`.

This is intentionally a different task from one-move SFT. Its validation reports set precision, recall, F1, exact-set match, and exact-order match.

In [ ]:
# Cell 1: mount Drive, copy the code locally, and install dependencies
import os
import shutil
import sys

from google.colab import drive

drive.mount('/content/drive')

# Upload the entire bundle folder to this Drive location, then edit only this path if needed.
DRIVE_BUNDLE_DIR = '/content/drive/MyDrive/colab_drive_bundle'
COLAB_CODE_DRIVE_DIR = DRIVE_BUNDLE_DIR
WORKSPACE = '/content/fen_move_all_moves_workspace'

if os.path.exists(WORKSPACE):
    shutil.rmtree(WORKSPACE)
os.makedirs(WORKSPACE, exist_ok=True)
if not os.path.isdir(COLAB_CODE_DRIVE_DIR):
    raise FileNotFoundError(f'Code folder not found: {COLAB_CODE_DRIVE_DIR}')

# Copy only executable project files; large data and checkpoints stay on Drive.
shutil.copytree(
    COLAB_CODE_DRIVE_DIR, os.path.join(WORKSPACE, 'fen_move_colab'),
    ignore=shutil.ignore_patterns('raw', 'datasets', 'runs', '__pycache__', '.ipynb_checkpoints'),
)
os.chdir(WORKSPACE)
sys.path.insert(0, WORKSPACE)
!bash fen_move_colab/install_colab.sh


In [ ]:
# Cell 2: choose the model and training settings
import os
import yaml

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN') or userdata.get('hf_token')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
except Exception:
    pass

# Set this to a Hugging Face model ID, a full model directory, or a PEFT adapter checkpoint.
BASE_MODEL_OR_CHECKPOINT = 'google/gemma-4-E4B-it'
# BASE_MODEL_OR_CHECKPOINT = f'{DRIVE_BUNDLE_DIR}/runs/previous_run/checkpoints/sft/step_100000'

# Keep v2 separate from the old run whose prompt incorrectly requested one move.
RUN_NAME = 'gemma4_e4b_all_legal_moves_v3'
DATASET_DIR = f'{DRIVE_BUNDLE_DIR}/datasets/lichess_legal_v1'
RUN_ROOT = f'{DRIVE_BUNDLE_DIR}/runs/{RUN_NAME}'
CHECKPOINT_DIR = f'{RUN_ROOT}/checkpoints/sft'
ALL_MOVES_TRAIN_PATH = f'{DATASET_DIR}/sft_all_legal_moves_v2_max80.jsonl'

# Complete all-moves lists can be long. This cap skips only unusually high-branching positions; it never truncates a target.
MAX_LEGAL_MOVES = 80
# Precision: 'bf16' (widest support), 'fp8' (L4/H100/Ada+), or 'nvfp4_qat' (Blackwell only).
# NVFP4 QAT uses the real NVFP4 numerical recipe through fake quantization; it does not provide native FP4 training speed.
PRECISION_MODE = 'bf16'
FP8_RECIPE = 'rowwise'  # 'tensorwise', 'rowwise', or 'rowwise_with_gw_hp'
FP8_EXPORT_GRANULARITY = 'tensorwise'  # 'tensorwise' is fastest/most compatible; 'rowwise' can improve accuracy.
FP8_EXPORT_SAVE_MERGED_BF16 = True  # Preserve a canonical model in addition to the FP8 deployment artifact.
FP8_EXPORT_VERIFY_RELOAD = True
FP8_EXPORT_OVERWRITE = False
USE_LORA = True

# Set every batch and performance value explicitly for the GPU you selected.
SFT_CHUNK_POSITIONS = 60000
MAX_SEQ_LENGTH = 512
NUM_TRAIN_EPOCHS = 5
PER_DEVICE_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03  # Converted to exact optimizer steps by the training script.
WARMUP_STEPS = None  # Set an integer to override WARMUP_RATIO.
LOGGING_STEPS = 20
CHECKPOINT_SAVE_STEPS = 300  # Optimizer steps between resumable Drive checkpoints.
CHECKPOINT_SAVE_TOTAL_LIMIT = None  # Keep every checkpoint for later evaluation.
VALIDATION_DURING_TRAINING = False  # Evaluate checkpoints later in the separate cell.
VALIDATION_SAMPLES = 128
VALIDATION_STEPS = 1000  # Used only if in-training validation is explicitly enabled.
VALIDATION_BATCH_SIZE = 8
VALIDATION_MAX_NEW_TOKENS = 512

# Performance controls. 'auto' uses full compile for Gemma 3 and safer regional compile for Gemma 4.
TORCH_COMPILE = True
TORCH_COMPILE_STRATEGY = 'auto'  # 'auto', 'full', or 'regional'
TORCH_COMPILE_MODE = 'default'  # 'default', 'reduce-overhead', 'max-autotune', or 'max-autotune-no-cudagraphs'
OPTIMIZER = 'adamw_torch_fused'
TF32 = True
GRADIENT_CHECKPOINTING = False  # Prefer this with Gemma 4 regional compile; enable only when VRAM requires it.
DATALOADER_NUM_WORKERS = 2

CONFIG = {
    'run': {'project_name': RUN_NAME, 'drive_root': RUN_ROOT, 'seed': 42},
    'data': {
        'sft_output': ALL_MOVES_TRAIN_PATH,
        'validation_output': f'{DATASET_DIR}/validation.jsonl',
    },
    'chunking': {'sft_chunk_samples': SFT_CHUNK_POSITIONS},
    'sft': {
        'model_id': BASE_MODEL_OR_CHECKPOINT,
        'output_mode': 'all_legal_moves',
        'max_seq_length': MAX_SEQ_LENGTH,
        'num_train_epochs': NUM_TRAIN_EPOCHS,
        'per_device_train_batch_size': PER_DEVICE_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'warmup_steps': WARMUP_STEPS,
        'logging_steps': LOGGING_STEPS,
        'resume_from_checkpoint': None,
        'checkpointing': {
            'enabled': True, 'save_steps': CHECKPOINT_SAVE_STEPS,
            'save_total_limit': CHECKPOINT_SAVE_TOTAL_LIMIT,
        },
        'validation': {
            'enabled': VALIDATION_DURING_TRAINING, 'samples': VALIDATION_SAMPLES, 'steps': VALIDATION_STEPS,
            'max_new_tokens': VALIDATION_MAX_NEW_TOKENS, 'batch_size': VALIDATION_BATCH_SIZE,
        },
    },
    'precision': {'mode': PRECISION_MODE},
    'performance': {
        'torch_compile': TORCH_COMPILE, 'torch_compile_strategy': TORCH_COMPILE_STRATEGY,
        'torch_compile_backend': 'inductor', 'torch_compile_dynamic': True,
        'torch_compile_mode': TORCH_COMPILE_MODE, 'optimizer': OPTIMIZER,
        'tf32': TF32, 'float32_matmul_precision': 'high',
        'gradient_checkpointing': GRADIENT_CHECKPOINTING,
        'gradient_checkpointing_use_reentrant': False,
        'dataloader_num_workers': DATALOADER_NUM_WORKERS, 'dataloader_pin_memory': True,
        'dataloader_persistent_workers': True, 'dataloader_prefetch_factor': 2,
    },
    'lora': {
        'enabled': USE_LORA, 'r': 32, 'alpha': 64, 'dropout': 0.05, 'bias': 'none',
        'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    },
    'fp8': {
        'enabled': PRECISION_MODE == 'fp8', 'recipe': FP8_RECIPE,
        'sft_max_seq_length': MAX_SEQ_LENGTH,
        'export': {
            'granularity': FP8_EXPORT_GRANULARITY,
            'save_merged_bf16': FP8_EXPORT_SAVE_MERGED_BF16,
            'verify_reload': FP8_EXPORT_VERIFY_RELOAD,
            'verification_max_new_tokens': 8, 'max_shard_size': '5GB',
            'modules_to_not_convert': ['lm_head'], 'overwrite': FP8_EXPORT_OVERWRITE,
        },
    },
    'nvfp4': {
        'enabled': PRECISION_MODE == 'nvfp4_qat', 'qat': True,
        'min_compute_capability_major': 10, 'min_compute_capability_minor': 0,
    },
}

CONFIG_PATH = 'fen_move_colab/all_legal_moves_runtime.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False)
print(f'Base model/checkpoint: {BASE_MODEL_OR_CHECKPOINT}')
print(f'Precision: {PRECISION_MODE}; compile: {TORCH_COMPILE}/{TORCH_COMPILE_STRATEGY}/{TORCH_COMPILE_MODE}; effective batch: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION}')
print(f'Run root: {RUN_ROOT}')
print(f'Checkpoints save automatically to: {CHECKPOINT_DIR}')


In [ ]:
# Cell 3: build the benchmark dataset from the bundled CSV if needed, then create the all-legal-moves target file
# This streams the expanded one-move SFT data and writes one lexicographically sorted UCI list per position.
# Set FORCE_REBUILD = True only if the source data or MAX_LEGAL_MOVES changed.
FORCE_REBUILD = False
SOURCE_SFT_PATH = f'{DATASET_DIR}/sft_train.jsonl'
if not os.path.exists(SOURCE_SFT_PATH):
    !python -m fen_move_colab.build_benchmark_datasets --puzzle-csv {DRIVE_BUNDLE_DIR}/raw/lichess_db_puzzle.csv.zst --output-dir {DATASET_DIR}


if FORCE_REBUILD or not os.path.exists(ALL_MOVES_TRAIN_PATH):
    !python -m fen_move_colab.build_all_legal_moves_dataset --source {SOURCE_SFT_PATH} --output {ALL_MOVES_TRAIN_PATH} --max-legal-moves {MAX_LEGAL_MOVES}
else:
    print(f'Reusing {ALL_MOVES_TRAIN_PATH}')


In [ ]:
# Cell 4: train the next ordered all-moves SFT chunk
# Re-run this cell to continue from the run's latest checkpoint automatically.
!python -m fen_move_colab.train_sft_chunk --config {CONFIG_PATH}


In [ ]:
# Cell 5: evaluate all intermediate and final checkpoints after training
# Results are saved after every checkpoint, so this evaluation can also be interrupted safely.
!python -m fen_move_colab.evaluate_sft_checkpoints --config {CONFIG_PATH} --checkpoint-root {CHECKPOINT_DIR} --samples {VALIDATION_SAMPLES} --batch-size {VALIDATION_BATCH_SIZE} --max-new-tokens {VALIDATION_MAX_NEW_TOKENS}


In [ ]:
# Cell 6: inspect the latest saved SFT state, training telemetry, and checkpoint evaluation report
import glob
import json

state_path = f'{RUN_ROOT}/state/sft_state.json'
if os.path.exists(state_path):
    print(json.dumps(json.load(open(state_path)), indent=2))
summary_paths = [p for p in glob.glob(f'{RUN_ROOT}/metrics/sft/step_*.json') if not p.endswith('_config.json')]
if summary_paths:
    path = max(summary_paths, key=os.path.getmtime)
    metrics = json.load(open(path))
    found_pct = metrics.get('fen_sft_val_all_moves_legal_moves_found_pct')
    if found_pct is not None:
        print(f'Available legal moves found by the model: {found_pct:.2f}%')
    gpu = (metrics.get('environment', {}).get('gpus') or [{}])[0]
    gib = 1024 ** 3
    print('\nREPORT SUMMARY')
    print(f"GPU: {gpu.get('name', 'CPU')}")
    print(f"Training runtime: {metrics.get('trainer_train_runtime', metrics.get('training_loop_seconds', 0)) / 60:.2f} minutes")
    input_tokens = metrics.get('fen_sft_cumulative_processed_input_tokens')
    supervised_tokens = metrics.get('fen_sft_cumulative_processed_supervised_tokens')
    if input_tokens is None:
        input_tokens = metrics.get('fen_sft_processed_input_tokens')
        supervised_tokens = metrics.get('fen_sft_processed_supervised_tokens')
    if input_tokens is not None:
        print(f"Training tokens: {input_tokens / 1_000_000:.3f}M non-padding input; {supervised_tokens / 1_000_000:.3f}M supervised")
        print(f"Input token throughput: {metrics.get('fen_sft_processed_input_tokens_per_second', 0):,.0f} tokens/s")
    print(f"Truncated samples/tokens: {metrics.get('fen_sft_truncated_samples', 0):,} / {metrics.get('fen_sft_truncated_tokens', 0):,}")
    print(f"Stage wall time: {metrics.get('stage_wall_seconds', 0) / 60:.2f} minutes")
    print(f"Peak allocated VRAM: {metrics.get('training_gpu_peak_allocated_bytes', 0) / gib:.2f} GiB")
    print(f"Peak reserved VRAM: {metrics.get('training_gpu_peak_reserved_bytes', 0) / gib:.2f} GiB")
    print(f"Trainable parameters: {metrics.get('model_parameters_trainable', 0):,} ({metrics.get('model_parameters_trainable_pct', 0):.3f}%)")
    print(f"Checkpoint size: {metrics.get('checkpoint_size_bytes', 0) / (1024 ** 2):.2f} MiB")
    print(json.dumps(metrics, indent=2))
evaluation_path = f'{RUN_ROOT}/metrics/sft/checkpoint_evaluation.json'
if os.path.exists(evaluation_path):
    evaluation = json.load(open(evaluation_path))
    print('\nBEST CHECKPOINT')
    print(json.dumps(evaluation.get('best', {}), indent=2))


In [ ]:
# Cell 7: merge the latest LoRA adapter and export persistent FP8 inference weights
# This preserves the adapter. With save_merged_bf16=True it also keeps the canonical merged BF16 model.
# The saved FP8 model dynamically quantizes activations and uses native FP8 weight/activation GEMMs.
#!python -m fen_move_colab.export_fp8_inference --config {CONFIG_PATH} --stage sft
